# Streamlined CRISPR Screen Analysis Tutorial

This tutorial demonstrates the essential workflow provided by `streamlined_crispr`.
We generate a small synthetic dataset using ``benchmarking/generate_demo_dataset.py``
so that quality control, effect-size estimation, and differential expression can be
executed end-to-end without external downloads.


## Prerequisites

Install the project in editable mode (with the optional `test` extras) to make
the package importable and provide AnnData, NumPy, pandas, and SciPy:

```bash
pip install -e .[test]
```

In [ ]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'src'))

from benchmarking.generate_demo_dataset import write_demo_dataset
import streamlined_crispr as scr

## Prepare the demo dataset

Generate the compact `.h5ad` file if it is missing and preview it with `read_h5ad_ondisk` to inspect key metadata without loading the full matrix. The helper returns a read-only `scr.AnnData` wrapper that automatically closes when it falls out of scope.


In [ ]:
data_path = Path('../data/demo_benchmark.h5ad')
output_dir = Path('tutorial_outputs')
output_dir.mkdir(exist_ok=True)

if not data_path.exists():
    print(f'Generating demo dataset at {data_path}')
    write_demo_dataset(data_path)

adata_ro = scr.read_h5ad_ondisk(data_path)

## Run quality control

Filtering keeps perturbations with adequate representation and removes low-quality cells and genes. When `control_label` or `gene_name_column` are omitted, the helper logs which defaults are used.


In [ ]:
filtered_ro = scr.pp.qc_summary(
    adata_ro,
    min_genes=5,
    min_cells_per_perturbation=5,
    min_cells_per_gene=5,
    perturbation_column='perturbation',
    output_dir=output_dir,
    data_name='tutorial',
)
filtered_ro


In [ ]:
qc_summary = {
    'total_cells': adata_ro.backed.n_obs,
    'kept_cells': filtered_ro.backed.n_obs,
    'total_genes': adata_ro.backed.n_vars,
    'kept_genes': filtered_ro.backed.n_vars,
}
qc_summary


## Compute effect sizes

Both effect-size estimators stream counts from disk and persist their outputs for inspection. They reuse the inferred control label and gene identifiers gathered during QC.


In [ ]:
adata_pb = scr.pb.average_log_expression(
    filtered_ro,
    perturbation_column='perturbation',
    output_dir=output_dir,
    data_name='tutorial_avg',
)
adata_pb.to_memory().to_df().head()


In [ ]:
adata_pseudo = scr.pb.pseudobulk(
    filtered_ro,
    perturbation_column='perturbation',
    output_dir=output_dir,
    data_name='tutorial_pseudo',
)
adata_pseudo.to_memory().to_df().head()


## Differential expression

Run both the Wald and Wilcoxon tests. Each call writes an AnnData file that contains the test statistics and automatically reuses the inferred control label.


In [ ]:
wald_de = scr.tl.rank_genes_groups(
    filtered_ro,
    perturbation_column='perturbation',
    method='wald',
    output_dir=output_dir,
    data_name='tutorial_wald',
)
wald_de.uns['rank_genes_groups']


In [ ]:
wilcoxon_de = scr.tl.rank_genes_groups(
    filtered_ro,
    perturbation_column='perturbation',
    method='wilcoxon',
    output_dir=output_dir,
    data_name='tutorial_wilcoxon',
)
wilcoxon_de.uns['rank_genes_groups']


## Inspect generated files

Every stage creates `.h5ad` artifacts for reproducibility and downstream analysis.


In [ ]:
sorted(path.name for path in output_dir.glob('*.h5ad'))

In [ ]:
filtered_ro.close()
adata_pb.close()
adata_pseudo.close()
wald_de.close()
wilcoxon_de.close()
adata_ro.close()
